# Radial distribution function — all systems & compositions

Lateral (in-plane) headgroup RDF between phosphate `P` atoms, for
**4 systems × 5 compositions × 3 lipid pairs**, plotted in the paper form:
one row of 3 panels per system (the two self-RDFs + the cross-RDF), with the
five compositions drawn as a light→dark gradient and shaded error bands.

Systems: DOPC/DOPA, DOPC/DOPG, DOPC/DLPG, DOPC/DLPA
Compositions (DOPC/anionic): 65/35, 72/28, 80/20, 87/13, 95/5

You can get the RDFs two ways — **GROMACS** (`rdf_gromacs.sh`, parsed below) or
**pure Python** (MDAnalysis, with block-averaged error bands). Both feed the same
plotting code.


## Setup

In [ ]:
# !pip install "MDAnalysis>=2.4" numpy matplotlib
%matplotlib inline
import os, itertools
import numpy as np
import matplotlib.pyplot as plt

## Config — systems, compositions, file layout

Point `files_for(system, comp)` at your data. Default layout:
`DOPC_DOPA/65_35/{sys.tpr, traj.xtc}` etc. Use the **reduced** headgroup
trajectory (`heads_skip.xtc` + `heads.tpr`) from `shrink_trajectory.sh` — it is
tiny and perfect for RDF.


In [ ]:
COMPS = ["65/35","72/28","80/20","87/13","95/5"]     # light -> dark

# per system: PC resname, anionic resname, colormap, panel labels
SYSTEMS = {
  "DOPC_DOPA": dict(pc="DOPC", x="DOPA", cmap="Reds",
                    labels=("PC-PC","PA-PA","PC-PA")),
  "DOPC_DOPG": dict(pc="DOPC", x="DOPG", cmap="Blues",
                    labels=("PC-PC","PG-PG","PC-PG")),
  "DOPC_DLPG": dict(pc="DOPC", x="DLPG", cmap="Greens",
                    labels=("PC-PC","PG-PG","PC-PG")),
  "DOPC_DLPA": dict(pc="DOPC", x="DLPA", cmap="Purples",
                    labels=("PC-PC","PA-PA","PC-PA")),
}

HEAD   = "P"
RMAX   = 4.0      # nm (paper x-axis)
NBINS  = 200
NBLOCK = 5        # time blocks -> error bands
STEP   = 1
OUTDIR = "results/rdf"; os.makedirs(OUTDIR, exist_ok=True)

def files_for(system, comp):
    c = comp.replace("/","_")
    base = f"{system}/{c}"
    return dict(top=f"{base}/heads.tpr", traj=f"{base}/heads_skip.xtc")

def pairs_for(system):
    s = SYSTEMS[system]
    return [(s["pc"], s["pc"]), (s["x"], s["x"]), (s["pc"], s["x"])]

## Python RDF — 2D lateral, block-averaged error bands

`compute_rdf_2d` returns `r (nm), g_mean, g_std`. Error bands are the standard
deviation of g(r) across `NBLOCK` contiguous time blocks. Needs a trajectory
(RDF is a time/ensemble average — a single frame is too noisy).

In [ ]:
import MDAnalysis as mda

def _pdist2d(a, b, Lx, Ly, same):
    dx = a[:,None,0]-b[None,:,0]; dy = a[:,None,1]-b[None,:,1]
    dx -= Lx*np.round(dx/Lx); dy -= Ly*np.round(dy/Ly)
    r = np.sqrt(dx*dx + dy*dy)
    return r[np.triu_indices(r.shape[0],1)] if same else r.ravel()

def compute_rdf_2d(top, traj, selA, selB, rmax=RMAX, nbins=NBINS,
                   nblock=NBLOCK, step=STEP):
    u = mda.Universe(top, traj)
    A = u.select_atoms(selA); B = u.select_atoms(selB)
    same = (selA == selB)
    rmaxA = rmax*10.0
    edges = np.linspace(0, rmaxA, nbins+1)
    ctr   = 0.5*(edges[1:]+edges[:-1])
    shell = np.pi*(edges[1:]**2 - edges[:-1]**2)
    frames = list(range(0, len(u.trajectory), step))
    blocks = np.array_split(frames, min(nblock, len(frames)))
    gb = []
    for blk in blocks:
        if len(blk) == 0: continue
        hist = np.zeros(nbins); norm = 0.0
        for fi in blk:
            ts = u.trajectory[fi]
            Lx, Ly = ts.dimensions[:2]; area = Lx*Ly
            a = A.positions[:, :2]; b = B.positions[:, :2]
            hist += np.histogram(_pdist2d(a, b, Lx, Ly, same), bins=edges)[0]
            Na, Nb = len(a), len(b)
            norm += 0.5*Na*((Nb-1)/area) if same else Na*(Nb/area)
        gb.append(hist/(norm*shell))
    gb = np.array(gb)
    return ctr/10.0, gb.mean(0), gb.std(0)

## Compute & save (Python route)

Runs every system/composition/pair that exists on disk and writes
`results/rdf/<system>_<comp>_<A>-<B>.dat` (columns: r_nm, g, g_std).

In [ ]:
def dat_path(system, comp, A, B):
    return f"{OUTDIR}/{system}_{comp.replace('/','_')}_{A}-{B}.dat"

for system in SYSTEMS:
    for comp in COMPS:
        f = files_for(system, comp)
        if not (os.path.exists(f["top"]) and os.path.exists(f["traj"])):
            print("skip (missing):", system, comp); continue
        for A, B in pairs_for(system):
            r, g, s = compute_rdf_2d(f["top"], f["traj"],
                                     f"name {HEAD} and resname {A}",
                                     f"name {HEAD} and resname {B}")
            np.savetxt(dat_path(system, comp, A, B),
                       np.column_stack([r, g, s]), header="r_nm g g_std")
            print("wrote", dat_path(system, comp, A, B))

## Alternative: parse the GROMACS `.xvg` files

If you ran `rdf_gromacs.sh` instead, read those `.xvg` outputs. (They carry the
mean g(r); use the Python route above if you want error bands.)

In [ ]:
def read_xvg(path):
    rows = [[float(x) for x in ln.split()]
            for ln in open(path) if not ln.startswith(("#","@"))]
    return np.array(rows)

def load_gmx(system, comp, A, B):
    c = comp.replace("/","_")
    d = read_xvg(f"{system}/{c}/rdf/rdf_{A}-{B}.xvg")
    return d[:,0], d[:,1], np.zeros_like(d[:,1])   # r_nm, g, (no band)

## Plot — paper form

`plot_system(system)` draws the 1×3 panel row with the 5 compositions as a
light→dark gradient, shaded ±std bands, a dashed reference line at g(r)=1, and
italic pair labels. Set `source="py"` (default, reads the `.dat` files) or
`source="gmx"` (reads the `.xvg` files).

In [ ]:
def _colors(cmap_name, n):
    cmap = plt.get_cmap(cmap_name)
    return [cmap(v) for v in np.linspace(0.35, 0.95, n)]

def _load(system, comp, A, B, source):
    if source == "gmx":
        return load_gmx(system, comp, A, B)
    d = np.loadtxt(dat_path(system, comp, A, B))
    return d[:,0], d[:,1], d[:,2]

def plot_system(system, source="py", ylim=(0.5,1.5), save=True):
    s = SYSTEMS[system]; pairs = pairs_for(system)
    colors = _colors(s["cmap"], len(COMPS))
    fig, axes = plt.subplots(1, 3, figsize=(9.2, 3.1))
    for ax, (A,B), lab in zip(axes, pairs, s["labels"]):
        for k, comp in enumerate(COMPS):
            try:
                r, g, sd = _load(system, comp, A, B, source)
            except (OSError, FileNotFoundError):
                continue
            ax.plot(r, g, color=colors[k], lw=1.4, label=comp)
            if np.any(sd):
                ax.fill_between(r, g-sd, g+sd, color=colors[k], alpha=0.18, lw=0)
        ax.axhline(1.0, color="0.6", lw=0.6, ls="--", zorder=0)
        ax.set_xlim(0, RMAX); ax.set_ylim(*ylim)
        ax.set_xlabel("Distance (nm)")
        ax.tick_params(direction="in", top=True, right=True)
        ax.text(0.96, 0.06, lab, transform=ax.transAxes, ha="right",
                style="italic", fontsize=13, fontweight="bold")
    axes[0].set_ylabel("RDF")
    h, l = axes[0].get_legend_handles_labels()
    if h:
        fig.legend(h, l, ncol=len(COMPS), frameon=True, loc="upper center",
                   bbox_to_anchor=(0.5, 1.08), columnspacing=1.0,
                   handlelength=1.4, fontsize=10,
                   title=f"{system.replace('_','/')}  (DOPC/X ratio)")
    fig.tight_layout()
    if save:
        out = f"{OUTDIR}/rdf_{system}.png"
        fig.savefig(out, dpi=200, bbox_inches="tight"); print("wrote", out)
    plt.show()

plt.rcParams.update({"font.family":"DejaVu Sans","axes.linewidth":1.0,
                     "xtick.major.size":4,"ytick.major.size":4})

## Make all four figures

In [ ]:
for system in SYSTEMS:
    plot_system(system, source="py")     # or source="gmx"

## Notes

- **2D vs 3D:** this computes the in-plane (lateral) RDF, matching `gmx rdf -xy`
  and the membrane convention. For a full 3D RDF, drop the `[:, :2]` slicing and
  the 2D shell normalisation.
- **Ordering / colours:** `COMPS` is listed light→dark (65/35 darkest-anionic to
  95/5). Change the colormap per system in `SYSTEMS` (Reds/Blues/Greens/Purples
  mirror the paper's per-panel colour families).
- **Error bands** are the std over `NBLOCK` time blocks — raise `NBLOCK` or run
  independent replicas for smoother, more rigorous bands.
- **y-limits:** the paper uses (0.5, 1.5) for self-RDFs and (0.75, 1.25) for
  cross panels; pass `ylim=` to `plot_system` to match a specific panel.
